### Step A — Create training examples  

We need pairs of: Instruction (human question) Output (correct Cypher query)

Example:
- Instruction: “List actors in Casino”
- Output: MATCH (m:Movie {title:'Casino'})<-[:ACTED_IN]-(p:Person) RETURN p.name;
Why: The model learns the pattern “question → Cypher”.

### Step B — Convert each example into ONE training text
- Models learn by predicting next tokens. So we join instruction and answer into a single text:

### Instruction:
List actors in the movie Casino.

### Response:
- MATCH (m:Movie {title:'Casino'})<-[:ACTED_IN]-(p:Person) RETURN p.name;
- Why: The model learns to “complete” the response after seeing instruction.

### Step C — Tokenize (convert words → numbers)
- Models don’t understand letters. They understand token IDs (numbers).
- Tokenizer turns the text into tokens.
Why: Training happens on token IDs.

### Step D — Load a base model (small, for learning)
- We’ll use a small model (distilgpt2) so it runs fast.
- Why: First make pipeline work. Later you can switch to a bigger chat model.

### Step E — Add LoRA adapters (the key idea)
- LoRA: freezes the big model adds a small trainable “patch” to some layers
- Why: cheap training, low memory.

Important LoRA knobs:
r (rank): adapter capacity (how much it can learn)
alpha: strength of LoRA update
dropout: prevents memorizing tiny dataset
target_modules: where to attach LoRA (for GPT2 it’s c_attn)

### Step F — Train only the adapters
We run a small training loop:
- send tokenized examples
- model predicts next tokens
- updates LoRA adapter weights only
- Why: base model stays intact.

### Step G — Save the adapter
We save only LoRA adapter (small files).
Later we load:
base model
adapter on top

### Step H — Test
Give a new question and see if it generates valid Cypher

1) Install libraries

In [ ]:
!pip -q install -U transformers datasets peft accelerate

2) Make your tiny Neo4j training data

In [ ]:
from datasets import Dataset

data = [
    {"instruction": "How many actors are there?",
     "output": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a);"},
    {"instruction": "Which actors played in the movie Casino?",
     "output": "MATCH (m:Movie {title:'Casino'})<-[:ACTED_IN]-(a:Person) RETURN a.name;"},
    {"instruction": "How many movies has Tom Hanks acted in?",
     "output": "MATCH (a:Person {name:'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m);"},
]

ds = Dataset.from_list(data)
ds

3) Choose model (small first)

4) Load tokenizer + create training text format

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def format_row(row):
    text = f"### Instruction:\n{row['instruction']}\n\n### Response:\n{row['output']}"
    return {"text": text}

ds_formatted = ds.map(format_row)
ds_formatted[0]["text"]

In [ ]:
# 5) Tokenize (text → numbers)
max_len = 256

# It takes text and converts it into numbers.
# "MATCH (m:Movie {title:'Casino'}) RETURN a.name;" 
""" {
  "input_ids": [50256, 1234, 5678, ...],
  "attention_mask": [1, 1, 1, ...]
}{
  "input_ids": [50256, 1234, 5678, ...],
  "attention_mask": [1, 1, 1, ...]
} """ 


def tokenize_fn(batch): 
    return tokenizer(batch["text"], truncation=True, max_length=max_len)

tokenized = ds_formatted.map(tokenize_fn, batched=True, remove_columns=ds_formatted.column_names)
tokenized

# output tokenized[0] 
""" {
  "input_ids": [15496, 1077, 2307, ...],
  "attention_mask": [1, 1, 1, ...]
} """

Why this step is CRITICAL for LoRA training

LoRA training still uses normal language model training.

The model learns:

“Given token 1, predict token 2… then token 3…”

So we must:

Combine instruction + answer into one sequence

Convert it to token IDs

Feed only token IDs to the trainer

This line does exactly that. 

In [ ]:
model_id = "distilgpt2"

In [ ]:
# 6) Load base model 
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_id)

In [ ]:
# 7) Attach LoRA adapters 
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],  # correct for GPT2
    bias="none"
)

model = get_peft_model(model, lora_config)
print(type(model))
model.print_trainable_parameters()

In [ ]:
# 8) Train (simple Trainer)

#TrainingArguments → configuration (knobs)
#Trainer → training loop (forward → loss → backward → update)
#DataCollatorForLanguageModeling → prepares batches correctly
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling


# examples have different lengths.pads them to the same length.creates labels for training
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
# mlm=True → BERT-style “fill the blank”
# mlm=False → GPT-style “predict next token”
# collator produces id → input_ids, attention_mask, labels 
# labels are what the model tries to predict 

training_args = TrainingArguments(
    output_dir="lora_demo_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=10,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

trainer.train()

""" 
1️⃣ Take batch from dataset
2️⃣ Pad sequences → same length
3️⃣ Feed input_ids to model
4️⃣ Model predicts next token
5️⃣ Compare prediction vs labels
6️⃣ Compute loss
7️⃣ Backpropagate loss
8️⃣ Update only LoRA weights
9️⃣ Repeat until dataset is done
🔁 Repeat for next epoch"""


In [ ]:
#v Step 9 — Save the LoRA adapter
model.save_pretrained("lora_adapter_v1")
tokenizer.save_pretrained("lora_adapter_v1")
import os
os.listdir("lora_adapter_v1")

In [ ]:
# Step 10 — Test the fine-tuned model 
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(model_id)
ft = PeftModel.from_pretrained(base, "lora_adapter_v1")
ft.eval() # We are not training. Be stable and deterministic. 

prompt = "### Instruction:\nList actors in the movie Casino.\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    out = ft.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

""" 
1️⃣ reads your prompt tokens
2️⃣ predicts the next token
3️⃣ appends it
4️⃣ repeats until:

60 tokens generated, or

end-of-sentence token """

print(tokenizer.decode(out[0], skip_special_tokens=True)) 
# [15496, 7, 182, ...] to MATCH (m:Movie ...) RETURN ... 
# skip_special_tokens=True removes tokens like: <eos> , <pad>


In [ ]:
print(type(model))
model.print_trainable_parameters()